# EDA – Gym Exercises Dataset

## Objectif

Vérifier que le fichier gym_exercises_dataset.csv est correctement chargé et obtenir une vue d’ensemble :

• dimensions du jeu de données (nombre de lignes et de colonnes) ;  
• typage des variables (champs textuels, listes de muscles, attributs techniques) ;  
• premières observations sur les valeurs manquantes et la présence éventuelle de doublons ;  
• cohérence générale du corpus avant analyse statistique et préparation NLP.


## Interprétation attendue

À partir de cette première inspection, il s’agira de commenter :

• la taille globale du dataset (nombre de lignes / colonnes) ;  
• la nature des variables (descriptifs textuels, attributs biomécaniques, catégories d’équipements) ;  
• la présence éventuelle de valeurs manquantes ou de champs textuels incomplets ;  
• l’existence (ou non) de doublons dans les données brutes ;  
• la cohérence générale des premières lignes (valeurs lisibles, descriptions complètes, muscles cohérents).

Cette interprétation sert de point de départ pour juger de la qualité du corpus et préparer les étapes suivantes : nettoyage, normalisation textuelle et structuration pour le traitement NLP du projet TrAIn.me.

## Configuration notebook

### Importation des librairies essentielles

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import prince

from pathlib import Path
import sys

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy import stats

# === Localiser automatiquement le package 'themes' en remontant l'arborescence ===
root = Path.cwd()  # ex : .../train.me/src/notebooks/eda/health_fitness_dataset
themes_root = None

for p in [root, *root.parents]:
    if (p / "themes").is_dir():
        themes_root = p
        break

if themes_root is None:
    raise FileNotFoundError(
        f"Impossible de trouver le dossier 'themes' en partant de {root}. "
        "Vérifie l'arborescence : il doit exister un dossier 'themes/' quelque part au-dessus."
    )

if str(themes_root) not in sys.path:
    sys.path.insert(0, str(themes_root))

print("✅ Ajouté au PYTHONPATH :", themes_root)
print("📂 Contenu de", themes_root, ":", [x.name for x in themes_root.iterdir()])

from themes.theme_train_me import set_trainme_theme

c:\Users\fback\Desktop\Projets\Dev\GitHub\train.me\.conda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Ajouté au PYTHONPATH : c:\Users\fback\Desktop\Projets\Dev\GitHub\train.me\src\notebooks
📂 Contenu de c:\Users\fback\Desktop\Projets\Dev\GitHub\train.me\src\notebooks : ['eda', 'model_training', 'preprocessing', 'themes', '__init__.py']


#### Configuration d’affichage et thème TrAIn.me


In [2]:
# Options d’affichage pour les tableaux
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# =============================
# Thème TrAIn.me (Seaborn/Matplotlib)
# =============================
USE_DARK_THEME = True  # bascule possible vers un thème clair si besoin

palette_trainme = set_trainme_theme(
    context="talk",
    font_scale=1.05,
    use_dark=USE_DARK_THEME
)

palette_trainme


['#00E5FF', '#00BFFF', '#0A84FF', '#1E90FF', '#00FFFF', '#64D8FF']

### Activation du thème TrAIn.me

In [3]:
if USE_DARK_THEME:
    palette_trainme = set_trainme_theme(context="talk", font_scale=1.05, use_dark=True)
else:
    sns.set_style("ticks")              # style clair sobre, axes renforcés
    sns.set_context("talk", font_scale=1.05)
    palette_trainme = sns.color_palette("deep")  # palette par défaut sobre
    print("🎨 Style activé : 'ticks' → sobre, axes renforcés (présentation pro)")

# Exemple de vérification :
print("Palette active :", palette_trainme.as_hex() if hasattr(palette_trainme, "as_hex") else palette_trainme)


Palette active : ['#00E5FF', '#00BFFF', '#0A84FF', '#1E90FF', '#00FFFF', '#64D8FF']


### 3.4.1. Chargement et inspection initiale

Cette section vise à charger le fichier gym_exercises_dataset.csv, vérifier ses dimensions
et obtenir un premier aperçu des données brutes. Elle reprend la même démarche que pour
les autres EDA du projet : contrôle du chargement, visualisation des premières lignes,
statistiques descriptives de base, analyse des valeurs manquantes et des doublons, puis
liste complète des colonnes disponibles.


#### Chargement du dataset gym_exercises_dataset.csv


In [4]:
# Téléchargement du dataset depuis Kaggle
dataset_path = kagglehub.dataset_download("rishitmurarka/gym-exercises-dataset")

# Recherche du fichier CSV dans le dossier téléchargé
csv_files = [f for f in os.listdir(dataset_path) if f.endswith(".csv")]
print("Fichiers trouvés :", csv_files)

# Construction du chemin complet vers le CSV
file_path = os.path.join(dataset_path, csv_files[0])


# Lecture du fichier CSV
df = pd.read_csv(file_path)

# Informations de confirmation
print("Dataset chargé avec succès !")
print(f"Dimensions : {df.shape[0]} lignes et {df.shape[1]} colonnes\n")


100%|██████████| 48.1k/48.1k [00:00<00:00, 574kB/s]

Extracting files...
Fichiers trouvés : ['gym_exercise_dataset.csv', 'stretch_exercise_dataset.csv']
Dataset chargé avec succès !
Dimensions : 617 lignes et 17 colonnes



#### Aperçu des premières lignes


In [5]:
df.head(10)


,Exercise Name,Equipment,Variation,Utility,Mechanics,Force,Preparation,Execution,Target_Muscles,Synergist_Muscles,Stabilizer_Muscles,Antagonist_Muscles,Dynamic_Stabilizer_Muscles,Main_muscle,Difficulty (1-5),Secondary Muscles,parent_id
0,Neck Flexion,Cable,No,Basic or Auxiliary,Isolated,Pull,Sit on bench facing away from middle pulley. P...,Move head away from pulley by bending neck for...,"Sternocleidomastoid,","None,","Rectus Abdominis, Obliques,",NaN,NaN,Neck,2,Sternocleidomastoid,NaN
1,Neck Flexion,Lever (plate loaded),No,Basic or Auxiliary,Isolated,Pull,Sit on seat in machine. Position padded lever ...,Move head forward by flexing neck until chin t...,"Sternocleidomastoid,","None,","Latissimus Dorsi, Deltoid, Posterior, Rhomboid...",NaN,NaN,Neck,2,Sternocleidomastoid,NaN
2,Lateral Neck Flexion,Lever (plate loaded),No,Auxiliary,Isolated,Pull,Sit on seat in machine with feet apart . Pos...,Move head down to side by laterally flexing ne...,"Sternocleidomastoid,","Splenius, Erector Spinae, Levator Scapulae, Tr...","Latissimus Dorsi, Pectoralis Major, Sternal, P...",NaN,NaN,Neck,2,"Sternocleidomastoid, Levator Scapulae",NaN
3,Neck Flexion,Lever (selectorized),No,Basic or Auxiliary,Isolated,Pull,Sit on seat in machine. Position padded lever ...,Move head forward by flexing neck until chin t...,"Sternocleidomastoid,","None,","Latissimus Dorsi, Deltoid, Posterior, Rhomboid...",NaN,NaN,Neck,2,Sternocleidomastoid,NaN
4,Lateral Neck Flexion,Lever (selectorized),No,Auxiliary,Isolated,Pull,Sit on seat in machine with feet apart. Positi...,Move head down to side by laterally flexing ne...,"Sternocleidomastoid,","Splenius, Erector Spinae, Levator Scapulae, Tr...","Latissimus Dorsi, Pectoralis Major, Sternal, P...",NaN,NaN,Neck,2,"Sternocleidomastoid, Levator Scapulae",NaN
5,Neck Flexion,Weighted,No,Basic or Auxiliary,Isolated,Pull,Place folded towel on weight plate. Lie supine...,Move head up by flexing neck until chin touche...,"Sternocleidomastoid,","None,","Rectus Abdominis, Obliques,",NaN,NaN,Neck,2,Sternocleidomastoid,NaN
6,Lateral Neck Flexion,Weighted,No,Auxiliary,Isolated,Pull,Place folded towel on weight plate. Lie on ben...,Move head up to side by laterally flexing neck...,"Sternocleidomastoid,","Splenius, Erector Spinae, Levator Scapulae, Tr...",NaN,NaN,NaN,Neck,2,"Sternocleidomastoid, Levator Scapulae",NaN
7,Wall Front Neck Bridge,Body Weight,No,Basic or Auxiliary,Isolated,Push & Pull,Place small or folded cushioned mat on wall or...,Roll down onto forehead until nose touches mat...,"Sternocleidomastoid,","Splenius, Trapezius, Upper, Levator Scapulae, ...","Rectus Abdominis, Obliques, Quadriceps,","Erector Spinae,",NaN,Neck,3,"Sternocleidomastoid, Upper Trapezius",NaN
8,Wall Side Neck Bridge,Body Weight,No,Auxiliary,Isolated,Pull,Place small or folded cushioned mat on column ...,Push side of head into mat and roll onto top o...,"Sternocleidomastoid,","Splenius, Trapezius, Upper, Levator Scapulae, ...","Gluteus Medius, Gluteus Minimus,",NaN,NaN,Neck,3,"Sternocleidomastoid, Upper Trapezius",NaN
9,Neck Flexion,Suspended,No,Basic or Auxiliary,Isolated,Pull,Place belt or strap around end of suspension t...,Increase angle of body by bowing head until ch...,"Sternocleidomastoid,","Splenius, Trapezius, Upper, Erector Spinae, Ce...",NaN,NaN,NaN,Neck,2,Sternocleidomastoid,NaN


#### Informations sur le typage, les valeurs manquantes et les doublons


In [6]:
# Typage et complétude globale
df_info = df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 617 entries, 0 to 616
Data columns (total 17 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Exercise Name               617 non-null    object 
 1   Equipment                   617 non-null    object 
 2   Variation                   613 non-null    object 
 3   Utility                     617 non-null    object 
 4   Mechanics                   617 non-null    object 
 5   Force                       617 non-null    object 
 6   Preparation                 617 non-null    object 
 7   Execution                   617 non-null    object 
 8   Target_Muscles              617 non-null    object 
 9   Synergist_Muscles           615 non-null    object 
 10  Stabilizer_Muscles          489 non-null    object 
 11  Antagonist_Muscles          141 non-null    object 
 12  Dynamic_Stabilizer_Muscles  265 non-null    object 
 13  Main_muscle                 617 non

### Statistiques descriptives des variables numériques

In [7]:
display(df.describe().T)

,count,mean,std,min,25%,50%,75%,max
Difficulty (1-5),617.0,2.737439,0.893215,1.0,2.0,3.0,3.0,5.0
parent_id,161.0,307.248447,158.649536,14.0,181.0,341.0,449.0,615.0


### Analyse des valeurs manquantes et doublons

In [8]:
# Comptage des valeurs manquantes par colonne
missing_counts = df.isna().sum().sort_values(ascending=False)

print("\n🔎 Valeurs manquantes par colonne (top 20) :")
display(missing_counts.head(20))

# Comptage des doublons ligne à ligne
n_duplicates = df.duplicated().sum()
print(f"\n🧬 Nombre de doublons (lignes strictement identiques) : {n_duplicates}")


🔎 Valeurs manquantes par colonne (top 20) :


Antagonist_Muscles            476
parent_id                     456
Dynamic_Stabilizer_Muscles    352
Stabilizer_Muscles            128
Variation                       4
Synergist_Muscles               2
Secondary Muscles               1
Difficulty (1-5)                0
Main_muscle                     0
Exercise Name                   0
Equipment                       0
Execution                       0
Preparation                     0
Force                           0
Mechanics                       0
Utility                         0
Target_Muscles                  0
dtype: int64


🧬 Nombre de doublons (lignes strictement identiques) : 0


#### Liste complète des colonnes disponibles


In [9]:
print("\n--- Liste des colonnes du dataset ---")
print(df.columns.tolist())



--- Liste des colonnes du dataset ---
['Exercise Name', 'Equipment', 'Variation', 'Utility', 'Mechanics', 'Force', 'Preparation', 'Execution', 'Target_Muscles', 'Synergist_Muscles', 'Stabilizer_Muscles', 'Antagonist_Muscles', 'Dynamic_Stabilizer_Muscles', 'Main_muscle', 'Difficulty (1-5)', 'Secondary Muscles', 'parent_id']
